In [2]:
from pathlib import Path
import sys

# Add parent directory to Python path
parent_dir = Path.cwd().parent
sys.path.append(str(parent_dir))

In [3]:
import geopandas as gpd 
import pandas as pd
import os 

import routing 
import osm 

import osmnx as ox
import co2

import networkx as nx


In [4]:
aoi = gpd.read_file("../aoi/aoi.gpkg")
aoi["id"] = aoi.index 
aoi

,Name,type,population,osmid,distance_to_node,geometry,id
0,Frankfurt Bezirk 27,Stadtteil,8908.140428,31941674,121.687367,"MULTIPOLYGON (((478966.135 5552679.23, 478964....",0
1,Frankfurt Bezirk 28,Stadtteil,6632.463686,31710439,33.489584,"MULTIPOLYGON (((479099.926 5552327.127, 479084...",1
2,Bockenheim,Stadtteil,32196.788691,11148720833,11.952396,"MULTIPOLYGON (((474137.825 5551457.352, 474134...",2
3,Bonames,Stadtteil,12635.036304,27405627,46.314649,"MULTIPOLYGON (((475759.938 5557724.818, 475757...",3
4,Frankfurt Bezirk 31,Stadtteil,1066.817343,415585886,68.327220,"MULTIPOLYGON (((477652.623 5550375.63, 477650....",4
...,...,...,...,...,...,...,...
94,Odenwaldkreis,Landkreis,96191.774804,269421067,1308.142115,"MULTIPOLYGON (((488539.291 5500844.059, 488539...",94
95,Fulda,Landkreis,227488.768518,1657720727,1397.636176,"MULTIPOLYGON (((550109.983 5578919.207, 550063...",95
96,Vogelsberg,Landkreis,102957.513896,352980044,6746.543470,"MULTIPOLYGON (((507642.328 5586973.323, 507488...",96
97,Rheinland-Pfalz,Bundesland,934385.526148,311638106,3289.258385,"MULTIPOLYGON (((489148.582 5483259.864, 489030...",97


In [5]:
streets_graph_path = os.path.normpath("../streets/streets.graphml")

In [6]:
G = ox.load_graphml(streets_graph_path)
nodes, edges = ox.graph_to_gdfs(G)
nodes = nodes.to_crs(aoi.crs)
edges = edges.to_crs(aoi.crs)
edges["travel_time_car"] = edges["travel_time_car"].astype(float)
G = ox.graph_from_gdfs(nodes,edges)

In [7]:
# no_go_area = gpd.read_file("../aoi/frankfurt_anlagenring.gpkg")
# no_go_area = no_go_area.to_crs(aoi.crs)
# edges = edges[~edges.intersects(no_go_area.union_all())]
# connected_nodes = set(edges.reset_index()["u"]).union(set(edges.reset_index()["v"]))
# nodes = nodes[nodes.index.isin(connected_nodes)]
# aoi = aoi[aoi["osmid"].isin(nodes.index)].reset_index(drop=True)
# G = ox.graph_from_gdfs(nodes,edges)

In [8]:
source_nodes = list(aoi.loc[aoi["type"].isin(["Stadtteil","Gemeinde"]),"osmid"])
target_nodes = list(aoi["osmid"])

In [23]:
travel_time_df = {"source":[],"target":[],"time":[],"co2":[],"path_nodes":[],"path_edges":[]}
for i in range(len(source_nodes)):
    for j in range(len(target_nodes)):
        if source_nodes[i] == target_nodes[j]:
            continue 
        
        print(source_nodes[i], " -> ",target_nodes[j])
        time, path = nx.single_source_dijkstra(
            G,
            source_nodes[i],
            target_nodes[j],
            weight="travel_time_car"
        )
        travel_time_df["source"].append(source_nodes[i])
        travel_time_df["target"].append(target_nodes[i])
        travel_time_df["time"].append(time)
        path_pairs = list(zip(path[:-1], path[1:]))
        mask = pd.MultiIndex.from_arrays(
            [
                edges.index.get_level_values("u"),
                edges.index.get_level_values("v"),
            ]
        ).isin(path_pairs)
        edges_path = edges.loc[mask,["travel_time_car","co2_car"]]
        edges_path = edges_path.loc[
            edges_path.groupby(level=["u", "v"])["travel_time_car"].idxmin()
        ]
        travel_time_df["co2"].append(edges_path["co2_car"].sum())
        travel_time_df["path_nodes"].append(list(path))
        travel_time_df["path_edges"].append(list(edges_path.index))


31941674  ->  31710439
31941674  ->  11148720833
31941674  ->  27405627
31941674  ->  415585886
31941674  ->  2610257828
31941674  ->  1230945944
31941674  ->  315814400
31941674  ->  565450
31941674  ->  304327239
31941674  ->  594809
31941674  ->  15514468
31941674  ->  1625070779
31941674  ->  4309483382
31941674  ->  561280
31941674  ->  29815683
31941674  ->  35790332
31941674  ->  5061970555
31941674  ->  1106687199
31941674  ->  30110936
31941674  ->  931209561
31941674  ->  35966670
31941674  ->  2926189911
31941674  ->  1071765467
31941674  ->  3296450583
31941674  ->  30836364
31941674  ->  30431291
31941674  ->  61104598
31941674  ->  561938
31941674  ->  256054082
31941674  ->  562041
31941674  ->  2303095483
31941674  ->  31081968
31941674  ->  812258958
31941674  ->  1975823607
31941674  ->  7631268429
31941674  ->  15514182
31941674  ->  10002637000
31941674  ->  267277112
31941674  ->  7640266285
31941674  ->  2165278485
31941674  ->  10929721132
31941674  ->  560668
31

KeyboardInterrupt: 

In [14]:
import numpy as np
import pandas as pd
import igraph as ig
from tqdm import tqdm

# =========================================================
# 1. KEEP BEST EDGE PER (u, v) BUT ALSO KEEP KEY
# =========================================================
edges_reset = edges.reset_index()

edges_best = edges_reset.loc[
    edges_reset.groupby(["u", "v"])["travel_time_car"].idxmin()
].copy()

u = edges_best["u"].to_numpy()
v = edges_best["v"].to_numpy()
k = edges_best["key"].to_numpy()   # 👈 KEEP KEY

travel_time = edges_best["travel_time_car"].to_numpy()
co2_vals = edges_best["co2_car"].to_numpy()

# =========================================================
# 2. NODE MAPPING
# =========================================================
nodes = np.unique(np.concatenate([u, v]))

node_to_idx = {n: i for i, n in enumerate(nodes)}
idx_to_node = {i: n for n, i in node_to_idx.items()}

sources = np.array([node_to_idx[x] for x in u])
targets = np.array([node_to_idx[x] for x in v])

# =========================================================
# 3. IGGRAPH BUILD
# =========================================================
g = ig.Graph(
    n=len(nodes),
    edges=list(zip(sources, targets)),
    directed=True
)

g.es["time"] = travel_time
g.es["co2"] = co2_vals

# IMPORTANT: store original metadata per edge
g.es["u"] = u
g.es["v"] = v
g.es["key"] = k   # 👈 CRITICAL

# =========================================================
# 4. TARGET INDEX
# =========================================================
target_idx = [node_to_idx[t] for t in target_nodes if t in node_to_idx]

# =========================================================
# 5. OUTPUT
# =========================================================
results = {
    "source": [],
    "target": [],
    "time": [],
    "co2": [],
    "path_nodes": [],
    "path_edges": []  # now includes (u,v,key)
}

# =========================================================
# 6. MAIN LOOP
# =========================================================
for source in tqdm(source_nodes, desc="Processing sources"):

    if source not in node_to_idx:
        continue

    s_idx = node_to_idx[source]

    dist = g.distances(source=s_idx, weights="time")[0]

    for t_idx in target_idx:

        target_name = idx_to_node[t_idx]

        if source == target_name:
            continue

        time = dist[t_idx]
        if time == float("inf"):
            continue

        path = g.get_shortest_paths(
            s_idx,
            to=t_idx,
            weights="time",
            output="vpath"
        )[0]

        path_nodes = [idx_to_node[i] for i in path]

        path_edges = []
        co2 = 0.0

        for i in range(len(path) - 1):
            u_idx = path[i]
            v_idx = path[i + 1]

            eid = g.get_eid(u_idx, v_idx, directed=True, error=False)

            if eid != -1:
                u_real = g.es[eid]["u"]
                v_real = g.es[eid]["v"]
                key_real = g.es[eid]["key"]

                path_edges.append((u_real, v_real, key_real))
                co2 += float(g.es[eid]["co2"])

        results["source"].append(source)
        results["target"].append(target_name)
        results["time"].append(time)
        results["co2"].append(co2)
        results["path_nodes"].append(path_nodes)
        results["path_edges"].append(path_edges)

# =========================================================
# 7. FINAL DATAFRAME
# =========================================================
travel_time_df = pd.DataFrame(results)

Processing sources: 100%|██████████| 84/84 [01:09<00:00,  1.21it/s]


In [16]:
travel_time_df

,source,target,time,co2,path_nodes,path_edges
0,31941674,31710439,535.569386,0.9045,"[31941674, 6061570821, 6061570824, 29954058, 2...","[(31941674, 6061570821, 0), (6061570821, 60615..."
1,31941674,11148720833,1951.417859,3.2857,"[31941674, 6061570821, 6061570824, 29954058, 2...","[(31941674, 6061570821, 0), (6061570821, 60615..."
2,31941674,27405627,1366.059983,2.7843,"[31941674, 6061570821, 6061570824, 29954058, 2...","[(31941674, 6061570821, 0), (6061570821, 60615..."
3,31941674,415585886,1220.721965,2.1734,"[31941674, 6061570821, 6061570824, 29954058, 2...","[(31941674, 6061570821, 0), (6061570821, 60615..."
4,31941674,2610257828,1124.519092,2.3985,"[31941674, 6061570821, 6061570824, 29954058, 2...","[(31941674, 6061570821, 0), (6061570821, 60615..."
...,...,...,...,...,...,...
8227,3159572406,269421067,6830.069213,20.3760,"[3159572406, 8428972952, 43659708, 6965500138,...","[(3159572406, 8428972952, 0), (8428972952, 436..."
8228,3159572406,1657720727,6366.014737,20.5447,"[3159572406, 8428972952, 43659708, 6965500138,...","[(3159572406, 8428972952, 0), (8428972952, 436..."
8229,3159572406,352980044,6255.433556,20.7522,"[3159572406, 8428972952, 43659708, 6965500138,...","[(3159572406, 8428972952, 0), (8428972952, 436..."
8230,3159572406,311638106,3233.058447,7.9744,"[3159572406, 8111781600, 43659691, 267055176, ...","[(3159572406, 8111781600, 0), (8111781600, 436..."


In [23]:
import igraph as ig
import pandas as pd
from tqdm import tqdm

# -----------------------------
# Build graph
# -----------------------------
edges_reset = edges.reset_index()

g = ig.Graph.TupleList(
    edges_reset[["u", "v"]].itertuples(index=False),
    directed=True
)

g.es["time"] = edges_reset["travel_time_car"].tolist()
g.es["co2"] = edges_reset["co2_car"].tolist()

# IMPORTANT: store real node IDs
# TupleList puts original IDs into g.vs["name"]
node_to_idx = {name: idx for idx, name in enumerate(g.vs["name"])}
idx_to_node = {idx: name for name, idx in node_to_idx.items()}

# -----------------------------
# Map sources/targets correctly
# -----------------------------
source_idx = [node_to_idx[n] for n in source_nodes]
target_idx = [node_to_idx[n] for n in target_nodes]

# -----------------------------
# Output
# -----------------------------
rows = []

# -----------------------------
# MAIN LOOP (84 sources)
# -----------------------------
for s in tqdm(source_idx):

    # shortest paths (by time)
    paths = g.get_shortest_paths(
        v=s,
        to=target_idx,
        weights="time",
        output="epath"
    )

    # iterate over targets
    for i, t in enumerate(target_idx):

        path_edges = paths[i]

        if not path_edges:
            continue

        # -----------------------------
        # reconstruct path + CO2
        # -----------------------------
        nodes_path = [s]
        edge_list = []
        co2 = 0.0
        time_val = 0.0

        for e in path_edges:
            edge = g.es[e]

            u = edge.source
            v = edge.target

            edge_list.append((u, v))
            nodes_path.append(v)

            co2 += float(edge["co2"])
            time_val += float(edge["time"])

        # convert back to original node IDs
        nodes_path_real = [idx_to_node[n] for n in nodes_path]
        edge_list_real = [
            (idx_to_node[u], idx_to_node[v]) for u, v in edge_list
        ]

        rows.append({
            "source": idx_to_node[s],
            "target": idx_to_node[t],
            "time": time_val,
            "co2": co2,
            "path_nodes": nodes_path_real,
            "path_edges": edge_list_real
        })

# -----------------------------
# FINAL DATAFRAME
# -----------------------------
travel_time_df = pd.DataFrame(rows)

100%|██████████| 84/84 [00:02<00:00, 33.20it/s]


In [24]:
travel_time_df

,source,target,time,co2,path_nodes,path_edges
0,31941674,31710439,535.569386,0.9045,"[31941674, 6061570821, 6061570824, 29954058, 2...","[(31941674, 6061570821), (6061570821, 60615708..."
1,31941674,11148720833,1951.417859,3.2857,"[31941674, 6061570821, 6061570824, 29954058, 2...","[(31941674, 6061570821), (6061570821, 60615708..."
2,31941674,27405627,1366.059983,2.7843,"[31941674, 6061570821, 6061570824, 29954058, 2...","[(31941674, 6061570821), (6061570821, 60615708..."
3,31941674,415585886,1220.721965,2.1734,"[31941674, 6061570821, 6061570824, 29954058, 2...","[(31941674, 6061570821), (6061570821, 60615708..."
4,31941674,2610257828,1124.519092,2.3985,"[31941674, 6061570821, 6061570824, 29954058, 2...","[(31941674, 6061570821), (6061570821, 60615708..."
...,...,...,...,...,...,...
8227,3159572406,269421067,6830.069213,20.3760,"[3159572406, 8428972952, 43659708, 6965500138,...","[(3159572406, 8428972952), (8428972952, 436597..."
8228,3159572406,1657720727,6366.014737,20.5447,"[3159572406, 8428972952, 43659708, 6965500138,...","[(3159572406, 8428972952), (8428972952, 436597..."
8229,3159572406,352980044,6255.433556,20.7522,"[3159572406, 8428972952, 43659708, 6965500138,...","[(3159572406, 8428972952), (8428972952, 436597..."
8230,3159572406,311638106,3233.058447,7.9744,"[3159572406, 8111781600, 43659691, 267055176, ...","[(3159572406, 8111781600), (8111781600, 436596..."


In [14]:
time, path = nx.single_source_dijkstra(
    G,
    31941674,
    316036463,
    weight="travel_time_car"
)

NetworkXNoPath: No path to 316036463.

In [15]:
m=nodes.loc[[31941674,
    316036463]].explore()
m.save("map.html")

In [6]:
edges.to_file("edges.gpkg")